# TogoLM Fine-Tuning — Google Colab

Fine-tune Mistral 7B Instruct on the TogoLM corpus using QLoRA.

**Requirements**: GPU runtime (T4 16GB or better)  
**Time**: ~2-3h on T4 for 500 examples, 3 epochs

Steps:
1. Install dependencies
2. Upload your `togolm_sft_train.jsonl` and `togolm_sft_eval.jsonl`
3. Run training
4. Push to HuggingFace Hub

In [ ]:
# 1. Install dependencies
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 \
    bitsandbytes==0.43.3 datasets==2.21.0 accelerate==0.34.2 \
    huggingface_hub

In [ ]:
# 2. Authenticate with HuggingFace (needed for Mistral gated model)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Upload your dataset files
# Run this cell, then use the file picker to upload:
#   - finetuning/datasets/togolm_sft_train.jsonl
#   - finetuning/datasets/togolm_sft_eval.jsonl
from google.colab import files
uploaded = files.upload()

In [ ]:
# 4. Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 5. Configuration
from dataclasses import dataclass, field

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "/content/togolm-7b"
TRAIN_FILE = "togolm_sft_train.jsonl"
EVAL_FILE  = "togolm_sft_eval.jsonl"

LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8   # effective batch = 16
LR = 2e-4
MAX_SEQ_LEN = 2048

In [ ]:
# 6. Load model with 4-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 7. Load dataset
import json
from datasets import Dataset, DatasetDict

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

ALPACA_TEMPLATE = """Below is an instruction about Togo. Write a response that answers it accurately.

### Instruction:
{instruction}

### Response:
{output}"""

def formatting_fn(example):
    return ALPACA_TEMPLATE.format(
        instruction=example['instruction'],
        output=example['output'],
    )

dataset = DatasetDict({
    'train': Dataset.from_list(read_jsonl(TRAIN_FILE)),
    'eval':  Dataset.from_list(read_jsonl(EVAL_FILE)),
})
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['eval'])}")

In [ ]:
# 8. Train
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    formatting_func=formatting_fn,
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()

In [ ]:
# 9. Save and push to HuggingFace Hub
HF_REPO = "your-username/togolm-7b"  # ← change this

trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

# Push LoRA adapter only (smaller upload, ~100MB vs 14GB)
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Model pushed to https://huggingface.co/{HF_REPO}')

In [ ]:
# 10. Quick inference test
from peft import PeftModel
from transformers import pipeline

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

question = "Quel est le taux d'imposition sur les sociétés au Togo ?"
prompt = f"### Instruction:\n{question}\n\n### Response:\n"
result = pipe(prompt, do_sample=False)
print(result[0]['generated_text'])